<a href="https://colab.research.google.com/github/stfnnnnnnn/karl-mangahas-flyrank/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfnnnnnnn/1st-act/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
pip -q install duckdb huggingface_hub

In [4]:
import os, getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass()

In [5]:
import duckdb

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )
    """
)


In [6]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients":
        f"read_parquet('{REL}/dim_clients.parquet')",

    "dim_content":
        f"read_parquet('{REL}/dim_content.parquet')",

    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",

    "fact_daily_sample":
        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",

    "fact_query_90d":
        f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [7]:
for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:22} {n:,} rows")

dim_clients            104 rows
dim_content            519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily             78,835,655 rows
fact_daily_sample      11,694,072 rows
fact_query_90d         2,414,248 rows


In [8]:
import pandas as pd

# Same CTE shape as W03 cell 20: client x content grain, prev30 / last30 windows,
# imp_prev30 >= 100 floor already baked in at the query level.
features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
),
windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
        SUM(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_last30,
        SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
        AVG(CASE WHEN f.report_date > b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END) AS pos_last30
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 60 DAY
      AND f.report_date <= b.end_d
    GROUP BY f.client_hash_id, f.content_hash_id
    HAVING imp_prev30 >= 100
)
SELECT * FROM windowed
""").df()

# Retrospective check only, exactly as in W03 — never a scoring input past this cell.
features["is_declining"] = (features["imp_last30"] < 0.8 * features["imp_prev30"]).astype(int)

print(features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(82025, 8)


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,clk_prev30,pos_last30,is_declining
0,client_3ffa76342f366962,content_456ab2db28595187,48.0,103.0,3.0,2.0,4.325833,1
1,client_3ffa76342f366962,content_5573434837db89c5,88.0,200.0,6.0,6.0,6.778892,1
2,client_3ffa76342f366962,content_7b17975c58745266,15.0,104.0,1.0,5.0,4.450000,1
3,client_3ffa76342f366962,content_b89167cd03d6ffc1,67.0,188.0,5.0,6.0,2.596627,1
4,client_e547b89c05043229,content_2e296120acb03e93,3115.0,706.0,0.0,0.0,60.204678,0


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words.**

A page deserves review first if it already demonstrated meaningful search demand in the previous 30-day feature window but is currently ranking outside page one. The underlying assumption is straightforward: if people were already finding and clicking the page, but it now occupies a weaker search position, then improving that page has a greater chance of recovering visibility than reviewing pages with little or no demand. Rather than predicting future decline directly, this baseline provides a transparent rule that prioritizes pages for earlier human review using only historical, leakage-safe signals.

**Signal 1 – Search position (position-based review logic)**

The first signal examines whether pages in weaker search position tiers are more likely to experience subsequent performance decline. Pages were grouped into the standard search position tiers (top_3, page_1, striking, page_3_5, and deep) and compared using the observed decline rate within each bucket. This analysis evaluates whether search position alone provides sufficient evidence to support the position-based component of the baseline rule.

**Verdict: MIXED**

Decline rates varied across the position tiers rather than increasing consistently as rankings became weaker. The highest observed decline rate occurred in the striking tier (26.2%, n = 17,366), while the page_3_5 (19.6%, n = 15,635) and deep (19.7%, n = 1,630) tiers showed lower decline rates. This suggests that search position contributes useful information but does not show a simple monotonic relationship with decline in this partition.


**Signal 2 – Average CTR across search position tiers**

The second signal examines how average click-through rate changes across different search position tiers. Since click-through rate naturally depends on ranking position, grouping pages by position provides a more meaningful comparison than evaluating CTR alone. This analysis verifies whether lower-ranked pages consistently receive lower click-through rates, providing evidence that search position is a useful signal for prioritizing review candidates.

**Verdict: CONFIRMED**
Average CTR decreased steadily from 0.352% (n = 6,527) in the top_3 tier to 0.083% (n = 1,630) in the deep tier, with each successive position tier showing lower observed click-through performance. This consistent trend supports the use of search position as a transparent signal for prioritizing review candidates.

**Reason code:** `visible_weak_position`, one code total, since this is the single-rule version and not the full multi-code audit.

**Action label:** `flag_for_review` when the score is above 0, otherwise `no_action`.

In [11]:
import numpy as np

pos_bins = [-1, 0, 3, 10, 20, 50, np.inf]
pos_labels = ["no_data", "top_3", "page_1", "striking", "page_3_5", "deep"]
features["position_tier"] = pd.cut(features["pos_last30"].fillna(0), bins=pos_bins, labels=pos_labels)
features["ctr_prev30"] = features["clk_prev30"] / features["imp_prev30"].replace(0, np.nan)

have_position = features[features["position_tier"] != "no_data"]

bucket_position = (
    have_position.groupby("position_tier", observed=True)["is_declining"]
        .agg(n="count", decline_rate="mean")
        .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
)
print("Signal 1 - position vs decline rate")
print(bucket_position)

bucket_ctr = (
    have_position.groupby("position_tier", observed=True)["ctr_prev30"]
        .agg(n="count", avg_ctr="mean")
        .reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
)
print("\nSignal 2 - position vs average CTR")
print(bucket_ctr)

Signal 1 - position vs decline rate
                   n  decline_rate
position_tier                     
top_3           6527      0.216179
page_1         37094      0.240524
striking       17366      0.261891
page_3_5       15635      0.196226
deep            1630      0.196933

Signal 2 - position vs average CTR
                   n   avg_ctr
position_tier                 
top_3           6527  0.003518
page_1         37094  0.003226
striking       17366  0.002606
page_3_5       15635  0.001800
deep            1630  0.000827


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The baseline intentionally remains simple and transparent. A page is only considered for review when it satisfies two fixed policy conditions: it must have demonstrated meaningful historical visibility (imp_prev30 ≥ 300) and it must currently rank outside page one (pos_last30 > 10). These thresholds are predetermined rather than learned from the data. Once both conditions are satisfied, previous 30-day impressions determine the ranking so that pages with greater demonstrated demand are reviewed first. No outcome-window information, future signals, or label-derived variables contribute to the score.

In [12]:
import os

features["is_visible"] = (features["imp_prev30"] >= 300).astype(int)
features["is_weak_position"] = (features["pos_last30"] > 10).astype(int)

features["baseline_score"] = (
    features["is_visible"] * features["is_weak_position"] * features["imp_prev30"]
)
features["reason_code"] = "visible_weak_position"
features["action"] = np.where(features["baseline_score"] > 0, "flag_for_review", "no_action")

ranked = features.sort_values("baseline_score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
out_cols = [
    "client_hash_id", "content_hash_id", "baseline_score", "reason_code", "action",
    "imp_prev30", "clk_prev30", "pos_last30", "is_declining",
]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

n_flagged = int((ranked["baseline_score"] > 0).sum())
base_rate = ranked["is_declining"].mean()
p20 = ranked["is_declining"].head(20).mean()
print(f"Wrote {len(ranked)} rows. {n_flagged} flagged for review ({n_flagged/len(ranked):.1%}).")
print(f"base rate: {base_rate:.1%} | precision@20: {p20:.1%}")
ranked[out_cols].head(10)

Wrote 82025 rows. 22968 flagged for review (28.0%).
base rate: 26.9% | precision@20: 15.0%


,client_hash_id,content_hash_id,baseline_score,reason_code,action,imp_prev30,clk_prev30,pos_last30,is_declining
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,224600.0,visible_weak_position,flag_for_review,224600.0,1.0,11.568483,1
1,client_23a62021009f63c4,content_e8a52cf3d5988c07,170127.0,visible_weak_position,flag_for_review,170127.0,654.0,14.968027,0
2,client_23a62021009f63c4,content_36e53e9c707674fc,108522.0,visible_weak_position,flag_for_review,108522.0,251.0,32.669102,0
3,client_23a62021009f63c4,content_df47d1b976106de4,90417.0,visible_weak_position,flag_for_review,90417.0,108.0,24.369431,0
4,client_fef1a8f436438636,content_84a6bf3578312e90,83691.0,visible_weak_position,flag_for_review,83691.0,78.0,20.804321,0
5,client_23a62021009f63c4,content_5e1c049f62e33b11,76602.0,visible_weak_position,flag_for_review,76602.0,122.0,18.113831,0
6,client_fef1a8f436438636,content_ba462518dad435fc,72994.0,visible_weak_position,flag_for_review,72994.0,41.0,27.451906,0
7,client_23a62021009f63c4,content_3df3f32f3fd58dea,71553.0,visible_weak_position,flag_for_review,71553.0,160.0,23.251180,0
8,client_23a62021009f63c4,content_b51957d7f4abe47e,60784.0,visible_weak_position,flag_for_review,60784.0,36.0,28.837608,0
9,client_23a62021009f63c4,content_bdf60c86117079be,54633.0,visible_weak_position,flag_for_review,54633.0,8.0,30.685334,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*


The baseline produces a ranked queue rather than a binary prediction, so the highest-ranked pages deserve manual inspection before drawing conclusions. For each of the top ten pages, I review why the rule selected it and identify one realistic situation where the recommendation could be misleading. This helps evaluate whether the rule is behaving as intended before comparing it against a machine learning model.

### **1**

- **Content:** `content_9c057b66c30a3abb`
- **Score:** 224600
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** The page had exceptionally high historical visibility (224,600 previous impressions) while ranking outside page one, indicating substantial demand that is not currently being captured by its search position.
- **What would make it wrong:** The keyword may be highly competitive, making the current ranking a realistic ceiling despite the strong historical demand.

---

### **2**

- **Content:** `content_e8a52cf3d5988c07`
- **Score:** 170127
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** Strong historical visibility combined with an average search position outside page one makes this page a reasonable review candidate.
- **What would make it wrong:** The current ranking may reflect strong competitors rather than problems with the page itself.

---

### **3**

- **Content:** `content_36e53e9c707674fc`
- **Score:** 108522
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** Historical demand remained high (108,522 previous impressions), but the page averaged position **32.67**, suggesting considerable search demand that is currently under-captured.
- **What would make it wrong:** The topic may naturally rank lower because of intense competition or declining search interest.

---

### **4**

- **Content:** `content_df47d1b976106de4`
- **Score:** 90417
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** The page exceeded the visibility threshold by a wide margin (90,417 previous impressions) while remaining around position **24.37**, making it a reasonable candidate for manual review.
- **What would make it wrong:** The lower ranking may be temporary and caused by seasonality or recent search algorithm changes.

---

### **5**

- **Content:** `content_84a6bf3578312e90`
- **Score:** 83691
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** The page generated **83,691 previous impressions** despite averaging position **20.80**, indicating meaningful historical demand that could justify additional investigation.
- **What would make it wrong:** The page may already be performing as expected for a highly competitive search query.

---

### **6**

- **Content:** `content_5e1c049f62e33b11`
- **Score:** 76602
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** With **76,602 historical impressions** and an average position of **18.11**, the page narrowly misses page-one visibility while continuing to attract meaningful search demand.
- **What would make it wrong:** Improving the ranking may have limited impact if competing pages consistently dominate the search results.

---

### **7**

- **Content:** `content_ba462518dad435fc`
- **Score:** 72994
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** The page accumulated nearly **73,000 previous impressions** while averaging position **27.45**, suggesting strong historical visibility despite relatively weak current rankings.
- **What would make it wrong:** The page may have been intentionally consolidated or deprioritized for reasons not visible in the dataset.

---

### **8**

- **Content:** `content_3df3f32f3fd58dea`
- **Score:** 71553
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** Historical impressions exceeded **71,000**, but the page continues to average position **23.25**, placing it well outside page one despite substantial search demand.
- **What would make it wrong:** The current ranking may already represent the page's realistic maximum performance.

---

### **9**

- **Content:** `content_b51957d7f4abe47e`
- **Score:** 60784
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** The page maintained more than **60,000 historical impressions** while averaging position **28.84**, indicating that meaningful search demand still exists despite limited visibility.
- **What would make it wrong:** User demand may have shifted toward newer or more authoritative competing pages.

---

### **10**

- **Content:** `content_bdf60c86117079be`
- **Score:** 54633
- **Action:** `flag_for_review`
- **Reason code:** `visible_weak_position`
- **Why:** The page remains above the visibility threshold with **54,633 previous impressions** but averages position **30.69**, making it a reasonable candidate under the baseline rule.
- **What would make it wrong:** The rule cannot distinguish between genuine optimization opportunities and intentional business decisions that affected the page's visibility.

In [13]:
top10_cols = [
    "client_hash_id", "content_hash_id", "baseline_score", "reason_code", "action",
    "imp_prev30", "clk_prev30", "pos_last30", "is_declining",
]
top10 = ranked.head(10)[top10_cols]
top10

,client_hash_id,content_hash_id,baseline_score,reason_code,action,imp_prev30,clk_prev30,pos_last30,is_declining
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,224600.0,visible_weak_position,flag_for_review,224600.0,1.0,11.568483,1
1,client_23a62021009f63c4,content_e8a52cf3d5988c07,170127.0,visible_weak_position,flag_for_review,170127.0,654.0,14.968027,0
2,client_23a62021009f63c4,content_36e53e9c707674fc,108522.0,visible_weak_position,flag_for_review,108522.0,251.0,32.669102,0
3,client_23a62021009f63c4,content_df47d1b976106de4,90417.0,visible_weak_position,flag_for_review,90417.0,108.0,24.369431,0
4,client_fef1a8f436438636,content_84a6bf3578312e90,83691.0,visible_weak_position,flag_for_review,83691.0,78.0,20.804321,0
5,client_23a62021009f63c4,content_5e1c049f62e33b11,76602.0,visible_weak_position,flag_for_review,76602.0,122.0,18.113831,0
6,client_fef1a8f436438636,content_ba462518dad435fc,72994.0,visible_weak_position,flag_for_review,72994.0,41.0,27.451906,0
7,client_23a62021009f63c4,content_3df3f32f3fd58dea,71553.0,visible_weak_position,flag_for_review,71553.0,160.0,23.251180,0
8,client_23a62021009f63c4,content_b51957d7f4abe47e,60784.0,visible_weak_position,flag_for_review,60784.0,36.0,28.837608,0
9,client_23a62021009f63c4,content_bdf60c86117079be,54633.0,visible_weak_position,flag_for_review,54633.0,8.0,30.685334,0


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Leakage check**

The baseline score uses only imp_prev30 and pos_last30, both measured before the evaluation window established in W03. Outcome-window variables such as imp_last30 and clk_last30 are never used to calculate the score. The is_declining label is created only after the ranking has been generated and is used solely to evaluate the baseline rather than influence it. The assertion below confirms that no outcome-window variables contribute to the rule..

**Weak picks**

1. Several highly ranked pages receive extremely large numbers of historical impressions but relatively few clicks. The rule prioritizes them because of their visibility, but impressions alone cannot distinguish between genuine opportunity and poor intent match.
2. Pages ranking between positions 20 and 40 are treated similarly even though some may target highly competitive keywords where further improvement is unrealistic. The baseline cannot account for keyword competitiveness.
3. The rule assumes historical visibility is always desirable. It cannot recognize pages that were intentionally deprioritized, recently consolidated, or affected by business decisions outside the available data.

In [14]:
# Leakage check: the rule's inputs must never include the label-adjacent columns
leakage_cols = {"imp_last30", "clk_last30", "is_declining"}
rule_input_cols = {"imp_prev30", "pos_last30"}
assert leakage_cols.isdisjoint(rule_input_cols), "Leakage: an outcome-window column is feeding the rule"
print("Clean: no imp_last30, clk_last30, or is_declining used as a rule input.")

weak_review_cols = [
    "client_hash_id", "content_hash_id", "imp_prev30", "clk_prev30",
    "pos_last30", "position_tier", "baseline_score",
]
ranked.head(20)[weak_review_cols]

Clean: no imp_last30, clk_last30, or is_declining used as a rule input.


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_last30,position_tier,baseline_score
0,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,224600.0,1.0,11.568483,striking,224600.0
1,client_23a62021009f63c4,content_e8a52cf3d5988c07,170127.0,654.0,14.968027,striking,170127.0
2,client_23a62021009f63c4,content_36e53e9c707674fc,108522.0,251.0,32.669102,page_3_5,108522.0
3,client_23a62021009f63c4,content_df47d1b976106de4,90417.0,108.0,24.369431,page_3_5,90417.0
4,client_fef1a8f436438636,content_84a6bf3578312e90,83691.0,78.0,20.804321,page_3_5,83691.0
5,client_23a62021009f63c4,content_5e1c049f62e33b11,76602.0,122.0,18.113831,striking,76602.0
6,client_fef1a8f436438636,content_ba462518dad435fc,72994.0,41.0,27.451906,page_3_5,72994.0
7,client_23a62021009f63c4,content_3df3f32f3fd58dea,71553.0,160.0,23.251180,page_3_5,71553.0
8,client_23a62021009f63c4,content_b51957d7f4abe47e,60784.0,36.0,28.837608,page_3_5,60784.0
9,client_23a62021009f63c4,content_bdf60c86117079be,54633.0,8.0,30.685334,page_3_5,54633.0


## Self-check

Before you submit, confirm each line honestly:

- [ - ] Every section above is filled — markdown thinking AND the code that backs it
- [ - ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ - ] No client names, URLs, or private queries anywhere
- [ - ] My claims use careful words: observed, measured, directional, decision-support
- [ - ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.